## Matching distance 

This is the notebook where we will merge all the code to compute the matching distance

In [23]:
import gudhi
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib
import matplotlib.patches as mpatches
from matplotlib import rc
import math
from math import cos, sin, pi, atan     
import itertools
from scipy.stats import qmc
import scipy.spatial
from scipy.optimize import linprog
import random as rnd
import sys 
sys.path.append('../Code/')
import modules as mod
import examples as ex
import plot as ourplot
import csv
import bunny_smaller
import omega_pts as omega
import cleanup_helper_funcs as chf
%load_ext autoreload
%autoreload 2
rc("font", **{"family": "serif", "serif": ["Times"]})
rc("text", usetex=True)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 0.1 Input for the case that M and N are both 1-dimensional 

(so we can sanity check with the equation)

In [24]:
tl_m = np.array([0.0, 6.0])
tr_m = np.array([6.0, 6.0])
bl_m = np.array([0.0, 0.0])
br_m = np.array([6.0, 0.0])

tl_n = np.array([4.0, 4.0])
tr_n = np.array([8.0, 4.0])
bl_n = np.array([4.0, 2.0])
br_n = np.array([8.0, 2.0])

critical_vals, nolub, M, N, critval_coords = chf.make_1dim_MN_example(tl_m, tr_m, bl_m, br_m, tl_n, tr_n, bl_n, br_n);

# note: current graph utility only works for more rectangles
graph = False
onedim_mods = True

## 0.2 todo: make helper func to make example for multiple rects

In [25]:
# critical_vals, nolub, M, N = ex.small_rectangle_example_cleaner_maybe()
# print("Critical values: ", critical_vals)

# TODO Still need this for general non 1D case

# put the modules into the correct format (eg like in examples.py)
# critval_coords = [] ## forgets the parent modules 
# for i in range(len(critical_vals)):
#     critval_coords.append(critical_vals[i].coords)
# print(critval_coords)

### 1. Identify the closures $\overline{C_M}$ and $\overline{C_N}$ 

### 2. Compute the set of switch-points $\Omega=\Omega(M,N)$ using the algorithms 

### 3. For each  line $L$ with positive slope that:
- passes through pairs of points in $\overline{C_M}\cup \overline{C_N}$, or
- passes through one point in $\overline{C_M}\cup \overline{C_N}$ and one point in ${\Omega}$,   or 
- passes through  one point of $\overline{C_M}\cup \overline{C_N}\cup{\Omega}$ and has diagonal slope

- ##### A. Compute the persistence diagrams $dgm(M^L)$, $dgm(N^L)$ of the restrictions of $M$ and $N$ to that line $L$;
- ##### B. Compute their weighted bottleneck distance $\hat m^L d_B(dgm(M^L),dgm(N^L))$

### 4. in the case of one-dim modules, check against Claudia's student's equation:

$$ \min ( \max ( \min \frac{b_i-a_i}{2} \min \frac{d_i-c_i}{2}  ), max( ||c-a||_\infty , ||d-b||_\infty   )  $$
- where $ R_M = (a1,b1)$ x $(a2,b2)$ and $R_N = (c1,d1)$ x $(c2,d2)$. 

In [28]:
## computes the least upper bound CM and CN separately 
# using the parent module information 
lub = omega.closure_lub(critical_vals) 

### run all three allgoriths 3v1, 2v2 paired , 2v2 unpaired
critpts, crits_no_lub, is_inf_2v2unp_orig, omegapts_3v1, is_inf_2v2p, orderedomegas_2v2unp = chf.compute_switch_points(critical_vals, nolub, makegraphyesno = graph, title = " ")
# combine the outputs together and deal with slopes vs points
omega_points_collection, omega_slopes_collection = chf.combine_omegas(is_inf_2v2unp_orig, omegapts_3v1, is_inf_2v2p, orderedomegas_2v2unp)

print("\n")
chf.compute_matching_dist(lub, critval_coords, omega_points_collection, omega_slopes_collection, M, N)


if onedim_mods:
    chf.check_onedim_mods(bl_m, tr_m, bl_n, tr_n)

3v1
**********
Computed 128 points (with duplicates)
Without duplicates 37
2v2paired
**********
Computed 19 points (with duplicates)
Without duplicates 11
2v2unp
**********
Computed 18 points (with duplicates)
Without duplicates 17


The matching distance is 3.0
Matching disstance is attained at the line with slope (1.0, 1.0) and intercept (0.0, 0.0)
matching distance (from equation) is 3.0
